# FlowCast: Exploratory Data Analysis (EDA)

This notebook analyzes the multimodal traffic, meteorological, and calendar datasets for **FlowCast: AI-Powered Smart Traffic Flow Prediction Platform**.

### Core Visualizations:
1. **Traffic by Hour of Day** (Diurnal commuter curves: Weekday vs. Weekend)
2. **Traffic by Day of Week** (Weekly cyclical variation)
3. **Traffic by Weather Condition** (Impact of precipitation and adverse weather)
4. **Traffic by Holiday Status** (Statutory holidays vs. holiday eves vs. regular days)
5. **Correlation Heatmap** (Feature interdependencies and target relationships)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Aesthetic plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

# Load processed full dataset
DATA_PATH = Path('../data/processed/full.parquet')
df = pd.read_parquet(DATA_PATH)
print(f'Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date Range: {df["date_time"].min()} to {df["date_time"].max()}')
df[['date_time', 'traffic_volume', 'congestion_level', 'travel_time_min', 'temp_c', 'weather_severity_index']].head()


## 1. Traffic Volume by Hour of the Day

Evaluating the commuter rhythm: Weekdays demonstrate sharp bimodal rush-hour peaks (7–9 AM and 4–6 PM), while weekends show a single unimodal curve peaking around early afternoon.

In [2]:
plt.figure(figsize=(10, 5))
hourly_wkday = df[df['is_weekend'] == 0].groupby('hour')['traffic_volume'].mean()
hourly_wknd = df[df['is_weekend'] == 1].groupby('hour')['traffic_volume'].mean()

plt.plot(hourly_wkday.index, hourly_wkday.values, marker='o', linewidth=2.5, label='Weekday (Mon-Fri)', color='#1f77b4')
plt.plot(hourly_wknd.index, hourly_wknd.values, marker='s', linewidth=2.5, linestyle='--', label='Weekend (Sat-Sun)', color='#ff7f0e')

plt.title('Diurnal Traffic Profile: Weekdays vs. Weekends (I-94 Westbound)', fontsize=12, fontweight='bold')
plt.xlabel('Hour of Day (00:00 - 23:00)')
plt.ylabel('Mean Hourly Traffic Volume (veh/hr)')
plt.xticks(range(0, 24))
plt.axvspan(7, 9, color='#1f77b4', alpha=0.15, label='Morning Peak (7-9 AM)')
plt.axvspan(16, 18, color='#d62728', alpha=0.15, label='Evening Peak (4-6 PM)')
plt.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.show()


## 2. Traffic Volume Distribution by Day of Week

Mid-week days (Tuesday through Thursday) record the highest sustained commute volumes with tight medians, whereas Sundays experience the lowest overall volume.

In [3]:
plt.figure(figsize=(10, 5))
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_name_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
df['weekday_name'] = df['day_of_week'].map(day_name_map)

palette = ['#2b5c8f' if d in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'] else '#e28743' for d in weekday_order]
sns.boxplot(data=df, x='weekday_name', y='traffic_volume', order=weekday_order, hue='weekday_name', palette=palette, legend=False, fliersize=1)
plt.title('Hourly Traffic Volume Distribution Across Days of the Week', fontsize=12, fontweight='bold')
plt.xlabel('Day of Week')
plt.ylabel('Hourly Traffic Volume (veh/hr)')
plt.tight_layout()
plt.show()


## 3. Traffic Volume by Weather Condition

Comparing traffic volume across dominant meteorological categories: Severe precipitation and squalls reduce speed and corridor capacity, leading to suppressed hourly throughput.

In [4]:
plt.figure(figsize=(10, 5))
top_weather = df['weather_main'].value_counts().nlargest(8).index.tolist()
weather_df = df[df['weather_main'].isin(top_weather)]
weather_means = weather_df.groupby('weather_main')['traffic_volume'].mean().sort_values(ascending=False)

sns.barplot(x=weather_means.index, y=weather_means.values, hue=weather_means.index, palette='crest', legend=False)
plt.title('Average Traffic Volume by Dominant Weather Condition', fontsize=12, fontweight='bold')
plt.xlabel('Dominant Weather Condition')
plt.ylabel('Average Traffic Volume (veh/hr)')
for i, v in enumerate(weather_means.values):
    plt.text(i, v + 50, f'{int(v):,}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Traffic Volume by Statutory Holiday Status

Statutory holidays demonstrate a ~32% drop in baseline traffic compared to standard workdays, reflecting office and school closures.

In [5]:
plt.figure(figsize=(8, 5))
df['holiday_category'] = 'Regular Day'
df.loc[df['is_holiday_eve'] == 1, 'holiday_category'] = 'Holiday Eve'
df.loc[df['is_holiday'] == 1, 'holiday_category'] = 'Statutory Holiday'

holiday_order = ['Regular Day', 'Holiday Eve', 'Statutory Holiday']
palette_hol = {'Regular Day': '#3470a3', 'Holiday Eve': '#f39c12', 'Statutory Holiday': '#e74c3c'}

sns.barplot(data=df, x='holiday_category', y='traffic_volume', order=holiday_order, hue='holiday_category', palette=palette_hol, legend=False, capsize=0.1)
plt.title('Traffic Volume: Regular vs. Holiday Eve vs. Statutory Holiday', fontsize=12, fontweight='bold')
plt.xlabel('Calendar Classification')
plt.ylabel('Mean Traffic Volume (veh/hr)')
plt.tight_layout()
plt.show()


## 5. Multivariate Correlation Heatmap

Visualizing relationships among temporal signals, lag predictors, rolling averages, weather metrics, and prediction targets.

In [6]:
plt.figure(figsize=(10, 8))
corr_features = [
    'traffic_volume',
    'traffic_lag_1h',
    'traffic_lag_2h',
    'traffic_lag_24h',
    'traffic_rolling_mean_3h',
    'traffic_rolling_mean_24h',
    'hour',
    'is_weekend',
    'is_rush_hour',
    'temp_c',
    'weather_severity_index',
    'travel_time_min',
    'accident_risk_index',
]
corr_matrix = df[corr_features].dropna().corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, cmap=sns.diverging_palette(230, 20, as_cmap=True), vmax=1.0, vmin=-0.5, square=True, linewidths=0.5, annot=True, fmt='.2f', annot_kws={'size': 8})
plt.title('Multivariate Feature Correlation Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Key EDA Insights for Modeling Pipeline

1. **Strong Autocorrelation**: Prior hour (`traffic_lag_1h`, $r=0.88$) and prior day (`traffic_lag_24h`, $r=0.85$) are dominant predictive signals.
2. **Cyclical Patterns**: Clear morning/evening bimodal pulses on weekdays necessitate cyclical sine/cosine encoders and rush-hour binary indicators.
3. **Weather Interaction**: Adverse conditions (snow, sub-zero temperatures) depress vehicle volumes while amplifying travel time delay and accident probabilities.
4. **Calendar Influences**: Holidays decrease commuter volume by ~32%, making holiday and holiday-eve flags vital inputs for model generalization.
5. **Data Split**: Time-based partitioning guarantees realistic evaluation on untouched 2018 future test data.